This notebook requires ApertureDB server and Python SDK 1.0. It asks arbitrary movie graph questions using live typed schema and validated native reads. Generated requests are limited to metadata Find/Get commands and 25 rows per command. The optional LLM provider receives the schema, question and query results; configure its API key before running.


## This is new section, where we try to define queries in Natural Language.

Aperturedb has a property graph interface in llama_index repo. will try to use that to query the KG.

In [ ]:
%pip install "openai>=1,<3"


In [ ]:
from aperturedb.CommonLibrary import create_connector
from aperturedb.Utils import Utils

client = create_connector()
utils = Utils(client)
utils.get_schema()


## Log internal queries

Sometimes to figure out the internal workings of the QA request/response it might be useful to figure out the inner workings. Run the following section to log the internal process.

In [ ]:
import os
from openai import OpenAI

# Uses OPENAI_API_KEY and an optional OPENAI_MODEL setting.
llm = OpenAI()
model_name = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")


## Build and query the ApertureDB Property graph

In [ ]:
"""General natural-language graph questions through server 1.0 native reads."""
import json
from types import SimpleNamespace


READ_COMMANDS = {
    'FindEntity', 'FindConnection', 'FindImage', 'FindVideo', 'FindBlob',
    'FindBoundingBox', 'FindPolygon', 'FindClip', 'FindFrame', 'FindDescriptor',
    'GetSchema', 'GetIndexes',
}
MAX_COMMANDS = 8
MAX_ROWS = 25


def validate_read_query(query):
    """Validate generated reads before execution; this never translates legacy queries."""
    if not isinstance(query, list) or not query or len(query) > MAX_COMMANDS:
        raise ValueError(f'Expected 1 to {MAX_COMMANDS} read commands')
    query = json.loads(json.dumps(query))
    for item in query:
        if not isinstance(item, dict) or len(item) != 1:
            raise ValueError('Each command must be a single-key object')
        command, body = next(iter(item.items()))
        if command not in READ_COMMANDS or not isinstance(body, dict):
            raise ValueError(f'Only native read commands are allowed: {command}')
        if any(key in body for key in ('set', 'batch', 'with_name', 'index_type', 'property_key')):
            raise ValueError('Use the server 1.0 query language')
        if body.get('blobs', False) is not False or 'k_neighbors' in body:
            raise ValueError('Graph questions use metadata reads without input/output blobs')
        if command.startswith('Find'):
            if command != 'FindConnection' and not isinstance(body.get('with_class'), str):
                raise ValueError('Object reads must select an explicit class')
            if body.get('constraints') is not None and not isinstance(body['constraints'], list):
                raise ValueError('Constraints must use expression arrays')
            def check_expression(expression):
                if not isinstance(expression, list):
                    return
                if len(expression) == 3 and expression[1] in ('in', 'not_in') and isinstance(expression[2], list):
                    raise ValueError('Membership lists need a json literal or array constructor')
                if expression and expression[0] in ('all', 'any', 'not'):
                    for child in expression[1:]:
                        check_expression(child)
            check_expression(body.get('constraints'))
            results = body.get('results', {'all_properties': True})
            if not isinstance(results, dict) or any(key in results for key in (
                    'limit', 'offset', 'sort', 'uniqueids', 'group_by_source', 'per_group')):
                raise ValueError('Pagination and grouping belong at command level')
            limit = body.get('limit', MAX_ROWS)
            if type(limit) is not int or not 1 <= limit <= MAX_ROWS:
                raise ValueError(f'limit must be between 1 and {MAX_ROWS}')
            offset = body.get('offset', 0)
            if type(offset) is not int or not 0 <= offset <= 1000:
                raise ValueError('offset must be between 0 and 1000')
            body['limit'] = limit
            body['results'] = results
            if command not in ('FindEntity', 'FindConnection'):
                body['blobs'] = False
    return query


NATIVE_CONTRACT = '''You generate ApertureDB 1.0 graph reads for arbitrary user questions.
Return only a JSON object with one key, "query", containing 1-8 native commands.
Use the provided live typed schema, preserving class/property/connection names.
Object reads use FindEntity/FindImage/FindBlob/FindVideo/FindBoundingBox/FindPolygon/
FindClip/FindFrame/FindDescriptor with explicit "with_class". Connections are
FindConnection; src/dst can refer to prior _ref selections. Customer graph
traversal uses "is_connected_to":{"ref":N,"connection_class":"HasCast"}.
A Movie→Professional HasCast relation is traversed in either direction by ref.
Declare a selection with _ref:N before another command references it.
Constraints are expression arrays: ["$name","==","Tom Hanks"], combined with
["all",expr1,expr2] or ["any",expr1,expr2]. Membership data uses
["$name","in",{"json":["Tom Hanks","Meg Ryan"]}]. Literal strings starting
with $ need an extra $ prefix. Property names inside constraints start with $.
Place limit, offset, sort, uniqueids and group_by_source at command level.
Use results:{"all_properties":true} or results:{"list":["title","budget"]}.
results:{"count":true} and scalar property aggregations are also available.
Each Find command has limit at most 25. Never write or remove data. Never use
set, batch, with_name, keyed constraints, input vectors, or binary output.
Example: [{"FindEntity":{"with_class":"Professional","_ref":1,
"constraints":["$name","==","Tom Hanks"],"results":{"list":["name"]}}},
{"FindEntity":{"with_class":"Movie","is_connected_to":{"ref":1,
"connection_class":"HasCast"},"results":{"list":["title","budget"]},"limit":25}}].
'''


class NativeGraphQA:
    """Generate validated native reads, then answer from their actual responses."""
    def __init__(self, database, llm_client, model):
        self.database, self.llm, self.model = database, llm_client, model

    def query(self, question):
        schema, _ = self.database.query([{'GetSchema': {}}])
        if not isinstance(schema, list) or schema[0]['GetSchema'].get('status') != 0:
            raise RuntimeError(f'Cannot read graph schema: {schema}')
        response = self.llm.chat.completions.create(
            model=self.model, temperature=0,
            response_format={'type': 'json_object'},
            messages=[{'role': 'system', 'content': NATIVE_CONTRACT},
                      {'role': 'user', 'content': json.dumps({
                          'question': question, 'schema': schema[0]['GetSchema']})}],
        )
        query = validate_read_query(json.loads(response.choices[0].message.content)['query'])
        rows, _ = self.database.query(query)
        if not isinstance(rows, list) or any(
                body.get('status') != 0 for item in rows for body in item.values()):
            raise RuntimeError(f'Generated graph read failed: {rows}')
        answer = self.llm.chat.completions.create(
            model=self.model, temperature=0,
            messages=[{'role': 'system', 'content':
                       'Answer using only the supplied database results. State when no records '
                       'match or the rows are limited to a sample; never invent missing facts.'},
                      {'role': 'user', 'content': json.dumps({
                          'question': question, 'query': query, 'results': rows,
                          'row_limit_per_command': MAX_ROWS})}],
        )
        return SimpleNamespace(response=answer.choices[0].message.content,
                               query=query, results=rows)


### Define the Schema to guide the LLM

In [ ]:
# The live typed schema supplies entities, properties and customer relations.
# Questions are not restricted to a fixed set of templates.


### Initialize the Property Graph Index, with an instance of SchemaLLMPathExtractor

In [ ]:
query_engine = NativeGraphQA(client, llm, model_name)


In [ ]:
def run_queries(query_engine):
    query_str = [
        ("Which movies is Tom Hanks associated with? Give me a list of movies.", True),
        ("Give me a list of movies starring Tom Hanks and Meg Ryan.", True),
        ("What is the genre of the movie 'Forrest Gump'?", True),
        ("Which other actors has Tom Hanks worked with?", True),
        ("What is the budget of the movie 'Forrest Gump'?", False),
        (
            "List all the crew members of the movie 'Forrest Gump', which were not actors.",
            False,
        ),
    ]
    for qs, _ in query_str:
        response = query_engine.query(qs)
        print(f"{qs=}")
        print(f"{response.response=}")
        print("=" * 50)


run_queries(query_engine)